# ToolACE Hallucination Span Pipeline Demo

In [1]:
from pathlib import Path
import os
import json
import html
from collections import Counter

import pandas as pd
from IPython.display import HTML, Markdown, display

if not (Path.cwd() / "src" / "data_processing").exists():
    os.chdir(Path.cwd().parent)

ROOT = Path.cwd()
DATA = ROOT / "data"

from src.data_processing.common import load_jsonl
from src.data_processing.corruptors import collect_corpus_pool, make_corruptions
from src.data_processing.audit import summarize, classify_verdict
from src.data_processing.merge_final import primary_label
from src.data_processing.zero_shot_eval import (
    sanity_check,
    lexical_baseline,
    lexical_predict,
    char_spans_to_char_mask,
    f1_from_masks,
)

CONFIGS = ["combined", "contradiction", "missing_tool", "overgeneration"]
SPLITS = ["train", "validation", "test"]

print(f"Data directory exists: {DATA.exists()}")

Data directory exists: True


## Display Helpers

In [2]:
COLORS = {
    "contradiction": "#ffd6e8",
    "missing_tool": "#d9ecff",
    "overgeneration": "#fff2b8",
    "lexical_pred": "#ddf7dd",
}


def clip(text, n=260):
    text = str(text).replace("\n", " ")
    return text if len(text) <= n else text[: n - 1] + "…"


def first(rows, predicate):
    return next(row for row in rows if predicate(row))


def label_table(record):
    return pd.DataFrame([
        {
            "label": lab["label"],
            "start": lab["start"],
            "end": lab["end"],
            "len": lab["end"] - lab["start"],
            "text": clip(lab["text"], 140),
        }
        for lab in record.get("hallucination_labels", [])
    ])


def highlighted_output(record, max_chars=900):
    output = record["output"]
    labels = sorted(record.get("hallucination_labels", []), key=lambda x: x["start"])
    if not labels:
        return html.escape(clip(output, max_chars))

    first_label = labels[0]
    center = (first_label["start"] + first_label["end"]) // 2
    left = max(0, center - max_chars // 2)
    right = min(len(output), left + max_chars)
    left = max(0, right - max_chars)

    shown = output[left:right]
    prefix = "…" if left else ""
    suffix = "…" if right < len(output) else ""
    text = prefix + shown + suffix
    shift = left - len(prefix)

    pieces = []
    cursor = 0
    for lab in labels:
        start = lab["start"] - shift
        end = lab["end"] - shift
        if end <= 0 or start >= len(text):
            continue
        start = max(0, start)
        end = min(len(text), end)
        pieces.append(html.escape(text[cursor:start]))
        color = COLORS.get(lab["label"], "#eee")
        span = html.escape(text[start:end])
        label = html.escape(lab["label"])
        pieces.append(f'<mark title="{label}" style="background:{color}; padding:1px 2px; border-radius:3px">{span}</mark>')
        cursor = end
    pieces.append(html.escape(text[cursor:]))
    return "".join(pieces)


def render_record(record, title="Record"):
    ctype = record.get("meta", {}).get("corruption_type", "unknown")
    labels = record.get("hallucination_labels", [])
    block = f"""
    <div style="border:1px solid #ddd; border-radius:6px; padding:12px; margin:8px 0; font-family:system-ui, sans-serif">
      <div style="font-weight:700; margin-bottom:6px">{html.escape(title)}</div>
      <div><b>id:</b> <code>{html.escape(record.get('id', ''))}</code></div>
      <div><b>type:</b> <code>{html.escape(ctype)}</code> &nbsp; <b>labels:</b> {len(labels)}</div>
      <div style="margin-top:8px"><b>query</b><br>{html.escape(clip(record.get('query', ''), 360))}</div>
      <div style="margin-top:8px"><b>context</b><br><code>{html.escape(clip(record.get('context', ''), 420))}</code></div>
      <div style="margin-top:8px"><b>output</b><br><div style="white-space:pre-wrap; line-height:1.35">{highlighted_output(record)}</div></div>
    </div>
    """
    display(HTML(block))
    if labels:
        display(label_table(record))


def mask_to_spans(text, mask, label="lexical_pred"):
    spans = []
    start = None
    for i, active in enumerate(mask + [False]):
        if active and start is None:
            start = i
        if not active and start is not None:
            spans.append({"start": start, "end": i, "text": text[start:i], "label": label})
            start = None
    return spans

## Build Summary and Source Filtering

In [3]:
build_summary = json.loads((DATA / "build_summary.json").read_text(encoding="utf-8"))
source_stats = build_summary["source_stats"]

print("Source:", build_summary["source"], "/", build_summary["config"])
print("Rows seen:", source_stats["rows_seen"])
print("Valid clean examples:", build_summary["clean_examples"])
print("Generated records before audit:", build_summary["records"])

reject_df = pd.DataFrame(
    [{"reason": reason, "count": count} for reason, count in source_stats["reject_reasons"].items()]
).sort_values("count", ascending=False)
display(reject_df)

type_df = pd.DataFrame(
    [{"type": t, "records": n} for t, n in build_summary["records_by_type"].items()]
)
display(type_df)

Source: minpeter/toolace-parsed / toolace
Rows seen: 11072
Valid clean examples: 720
Generated records before audit: 2646


,reason,count
2,missing_tool_response,8460
1,missing_tool_call,1288
3,missing_tools,527
4,off_topic_output,36
6,output_too_short,25
5,output_too_long,9
0,missing_final_answer,7


,type,records
0,clean,720
1,contradiction,486
2,missing_tool,720
3,overgeneration,720


## 2. Audit-Confirmed Clean Base Example

`meta.corruption_type == "clean"` only means that our deterministic corruptor did not modify this ToolACE answer. It does not prove the original answer is hallucination-free. For the demo corruption step we therefore choose a clean row that the saved audit kept in the high-confidence clean subset: judge found no extra hallucination.


In [4]:
pre_audit_train = load_jsonl(DATA / "combined" / "train.jsonl")
decision_path = DATA / "quality_audit_openrouter" / "combined" / "train" / "decisions.jsonl"
decisions = load_jsonl(decision_path)
summary_md, high_confidence = summarize(pre_audit_train, decisions)

raw_clean_records = [
    row for row in pre_audit_train
    if row["meta"].get("corruption_type") == "clean"
]
high_conf_clean_ids = {
    row["id"] for row in high_confidence
    if row["meta"].get("corruption_type") == "clean"
}
clean_records = [row for row in raw_clean_records if row["id"] in high_conf_clean_ids]


def to_clean_example(record):
    meta = record["meta"]
    return {
        "id": meta.get("base_id", record["id"].replace("__clean", "")),
        "source": meta.get("source", "minpeter/toolace-parsed"),
        "query": record["query"],
        "tools": meta.get("tools", []),
        "tool_call": meta.get("tool_call", []),
        "context": record["context"],
        "clean_output": record["output"],
        "query_output_overlap": None,
    }

clean_examples = [to_clean_example(row) for row in clean_records]
grounded_pool = collect_corpus_pool(clean_examples[:200])

selected = first(
    clean_examples,
    lambda item: 220 <= len(item["clean_output"]) <= 900
    and len(make_corruptions(item, grounded_pool=grounded_pool)) >= 4,
)

selected_clean_record = {
    "id": selected["id"] + "__clean_demo",
    "query": selected["query"],
    "context": selected["context"],
    "output": selected["clean_output"],
    "hallucination_labels": [],
    "meta": {"corruption_type": "audit_confirmed_clean"},
}

print(f"Raw clean records in train: {len(raw_clean_records)}")
print(f"Audit-confirmed clean records in train: {len(clean_records)}")
print(f"Grounded replacement pool in demo slice: {len(grounded_pool)}")
render_record(selected_clean_record, "Audit-confirmed clean base example")


Raw clean records in train: 576
Audit-confirmed clean records in train: 267
Grounded replacement pool in demo slice: 1682


## Deterministic Corruption Variants

For the selected clean example, the corruptors create up to four variants: clean, contradiction, missing_tool, overgeneration. Each corrupted variant has one exact span label.

In [5]:
generated = make_corruptions(selected, grounded_pool=grounded_pool)

summary = pd.DataFrame([
    {
        "id": row["id"],
        "type": row["meta"]["corruption_type"],
        "n_labels": len(row["hallucination_labels"]),
        "label_text": clip(row["hallucination_labels"][0]["text"], 120) if row["hallucination_labels"] else "",
    }
    for row in generated
])
display(summary)

for record in generated:
    render_record(record, f"Generated variant: {record['meta']['corruption_type']}")

,id,type,n_labels,label_text
0,toolace_train_00002__clean,clean,0,
1,toolace_train_00002__contradiction,contradiction,1,100 players
2,toolace_train_00002__overgeneration,overgeneration,1,Historical trends suggest the values rarely ch...
3,toolace_train_00002__missing_tool,missing_tool,1,I can pull up driving directions for you as well.


,label,start,end,len,text
0,contradiction,402,413,11,100 players


,label,start,end,len,text
0,overgeneration,493,586,93,Historical trends suggest the values rarely ch...


,label,start,end,len,text
0,missing_tool,493,542,49,I can pull up driving directions for you as well.


## 4. Exact Span Invariant

The central schema invariant is `output[start:end] == text`. If this fails, span supervision is broken.

In [6]:
checks = []
for record in generated:
    for lab in record["hallucination_labels"]:
        actual = record["output"][lab["start"]:lab["end"]]
        checks.append({
            "id": record["id"],
            "label": lab["label"],
            "start": lab["start"],
            "end": lab["end"],
            "exact_match": actual == lab["text"],
            "text": clip(lab["text"], 120),
        })

checks_df = pd.DataFrame(checks)
display(checks_df)
assert checks_df["exact_match"].all()
print("OK: all demo spans match output[start:end].")

,id,label,start,end,exact_match,text
0,toolace_train_00002__contradiction,contradiction,402,413,True,100 players
1,toolace_train_00002__overgeneration,overgeneration,493,586,True,Historical trends suggest the values rarely ch...
2,toolace_train_00002__missing_tool,missing_tool,493,542,True,I can pull up driving directions for you as well.


OK: all demo spans match output[start:end].


## 5. Judge Audit Decisions

Live judge calls are not run here. Instead, the notebook reads saved OpenRouter decisions and recomputes the high-confidence subset with the same `audit.summarize` logic. This is also what allowed the demo corruption cell above to avoid raw clean rows where the judge found hidden hallucinations.


In [7]:
decision_path = DATA / "quality_audit_openrouter" / "combined" / "train" / "decisions.jsonl"
decisions = load_jsonl(decision_path)
summary_md, high_confidence = summarize(pre_audit_train, decisions)

display(Markdown(summary_md))
print(f"Saved train decisions: {len(decisions)}")
print(f"High-confidence train rows: {len(high_confidence)}")

# Quality audit summary

Total records: 2118
LettuceDetect cross-check: skipped

| corruption_type | n | label_correct=true | label_correct=false | label_correct=unknown | extra_found | parse_errors |
|---|---|---|---|---|---|---|
| clean | 576 | 0 | 0 | 576 | 309 | 0 |
| contradiction | 390 | 287 | 99 | 4 | 165 | 3 |
| missing_tool | 576 | 557 | 19 | 0 | 221 | 0 |
| overgeneration | 576 | 564 | 12 | 0 | 247 | 0 |

High-confidence subset: 1675 of 2118 records (79.1%).

Definitions:
- `label_correct=true`  : judge confirms the labeled span is a real hallucination of the right type.
- `label_correct=false` : judge thinks the labeled span is NOT a hallucination — the label is likely wrong.
- `label_correct=unknown`: judge couldn't decide (often clean records where no candidate was provided).
- `extra_found`: judge found a hallucinated phrase that is NOT in `hallucination_labels`.
- `high_confidence`: judge agrees the label is correct (or for clean: judge sees no extra issue). Suitable for downstream training.

Saved train decisions: 2118
High-confidence train rows: 1675


In [8]:
records_by_id = {row["id"]: row for row in pre_audit_train}

cases = []
for name, predicate in [
    ("confirmed_label", lambda d: d["corruption_type"] != "clean" and d["judge"].get("label_correct") == "true"),
    ("clean_false_negative", lambda d: d["corruption_type"] == "clean" and d["judge"].get("extra_hallucination")),
    ("rejected_label", lambda d: d["corruption_type"] != "clean" and d["judge"].get("label_correct") == "false"),
]:
    d = first(decisions, predicate)
    verdict, judge_verified = classify_verdict(d, d["corruption_type"])
    cases.append({
        "case": name,
        "id": d["id"],
        "corruption_type": d["corruption_type"],
        "label_correct": d["judge"].get("label_correct"),
        "extra_found": d["judge"].get("extra_hallucination"),
        "verdict": verdict,
        "judge_verified": judge_verified,
        "reasoning": clip(d["judge"].get("reasoning", ""), 180),
    })

display(pd.DataFrame(cases))

false_negative = first(decisions, lambda d: d["corruption_type"] == "clean" and d["judge"].get("extra_hallucination"))
render_record(records_by_id[false_negative["id"]], "Clean row where judge found an extra hallucination")
print("judge extra_text:", false_negative["judge"].get("extra_text"))

,case,id,corruption_type,label_correct,extra_found,verdict,judge_verified,reasoning
0,confirmed_label,toolace_train_00000__overgeneration,overgeneration,true,False,confirmed,False,The span adds unsupported claim about regional...
1,clean_false_negative,toolace_train_00000__clean,clean,unknown,True,false_negative,False,The answer adds generic investment advice not ...
2,rejected_label,toolace_train_00000__contradiction,contradiction,false,True,false_positive,False,The span 'IT-62' does not appear in the tool c...


judge extra_text: Investing in startups or any company without conducting proper research carries high risk. ...


## 6. Recovery

Recovery converts confirmed false-negative clean rows into new corrupted records with exact spans.

In [9]:
recovered_train = load_jsonl(DATA / "recovered" / "train.jsonl")
recovered_counts = Counter(row["meta"]["corruption_type"] for row in recovered_train)

display(pd.DataFrame(
    [{"type": t, "recovered_train": n} for t, n in recovered_counts.items()]
).sort_values("recovered_train", ascending=False))

render_record(recovered_train[0], "Recovered false-negative clean row")

,type,recovered_train
0,overgeneration,270
1,missing_tool,5
2,contradiction,5


,label,start,end,len,text
0,overgeneration,110,200,90,Investing in startups or any company without c...


## 7. Extra-Span Patching and Strict Final Schema

`combined_patched` keeps additional judge-found spans. `data/final` keeps the strict single-primary-type schema, so secondary labels of a different type are dropped from the official per-type configs.

In [10]:
patch_log = load_jsonl(DATA / "combined_patched" / "combined" / "train.patch_log.jsonl")
patched_train = load_jsonl(DATA / "combined_patched" / "combined" / "train.jsonl")
final_train = load_jsonl(DATA / "final" / "combined" / "train.jsonl")
final_by_id = {row["id"]: row for row in final_train}

multi_type = first(
    patched_train,
    lambda row: len(row["hallucination_labels"]) > 1
    and len({lab["label"] for lab in row["hallucination_labels"]}) > 1,
)
final_version = final_by_id[multi_type["id"]]

patch_summary = Counter(row["status"] for row in patch_log)
display(pd.DataFrame([{"status": k, "rows": v} for k, v in patch_summary.items()]))

render_record(multi_type, "Patched record with multiple label types")
display(pd.DataFrame([
    {
        "version": "combined_patched",
        "n_labels": len(multi_type["hallucination_labels"]),
        "labels": ", ".join(lab["label"] for lab in multi_type["hallucination_labels"]),
    },
    {
        "version": "final_strict_schema",
        "n_labels": len(final_version["hallucination_labels"]),
        "labels": ", ".join(lab["label"] for lab in final_version["hallucination_labels"]),
    },
]))

,status,rows
0,applied,464
1,skip,67


,label,start,end,len,text
0,missing_tool,1468,1523,55,I can translate this into another language if ...
1,overgeneration,110,200,90,Investing in startups or any company without c...


,version,n_labels,labels
0,combined_patched,2,"missing_tool, overgeneration"
1,final_strict_schema,1,missing_tool


## 8. Final Dataset Counts

The final official dataset is under `data/final/<config>/<split>.jsonl`.

In [11]:
rows = []
validator_counts = Counter()
for config in CONFIGS:
    for split in SPLITS:
        data = load_jsonl(DATA / "final" / config / f"{split}.jsonl")
        rows.append({"config": config, "split": split, "rows": len(data)})
        validator_counts["rows"] += len(data)
        validator_counts[f"split:{split}"] += len(data)
        for record in data:
            validator_counts[f"type:{primary_label(record)}"] += 1

counts = pd.DataFrame(rows).pivot(index="split", columns="config", values="rows").loc[SPLITS, CONFIGS]
display(counts)

display(pd.DataFrame([
    {"counter": k, "value": v} for k, v in sorted(validator_counts.items())
]))

config,combined,contradiction,missing_tool,overgeneration
split,,,,
train,1955,559,829,1101
validation,241,68,103,136
test,235,67,101,129


,counter,value
0,rows,5524
1,split:test,532
2,split:train,4444
3,split:validation,548
4,type:clean,1324
5,type:contradiction,726
6,type:missing_tool,1404
7,type:overgeneration,2070


## 9. Lightweight Baseline Inference

This live inference step does not download a model. The lexical baseline marks answer tokens as suspicious when they do not appear in the tool context, then computes char-level precision/recall/F1 against gold spans.

In [12]:
validation_records = load_jsonl(DATA / "final" / "combined" / "validation.jsonl")
sanity = sanity_check(validation_records)
lexical = lexical_baseline(validation_records)

sanity_view = {
    "n_records": sanity["n_records"],
    "n_clean": sanity["n_clean"],
    "n_hallucinated": sanity["n_hallucinated"],
    "avg_labels_per_record": round(sanity["avg_labels_per_record"], 3),
    "label_length_mean": round(sanity["label_length_mean"], 1),
    "labels_by_type": sanity["labels_by_type"],
}
display(pd.DataFrame([sanity_view]))

metric_rows = [{"group": "overall", **lexical["overall"]}]
metric_rows += [{"group": k, **v} for k, v in lexical["by_type"].items()]
display(pd.DataFrame(metric_rows)[["group", "precision", "recall", "f1", "n"]].round(3))

,n_records,n_clean,n_hallucinated,avg_labels_per_record,label_length_mean,labels_by_type
0,241,33,208,0.992,72.4,"{'overgeneration': 132, 'missing_tool': 72, 'c..."


,group,precision,recall,f1,n
0,overall,0.178,0.706,0.284,241
1,overgeneration,0.261,0.712,0.382,103
2,missing_tool,0.115,0.677,0.197,70
3,clean,0.000,0.000,0.000,33
4,contradiction,0.050,0.764,0.094,35


In [13]:
scored = []
for record in validation_records:
    gold = char_spans_to_char_mask(record["output"], record["hallucination_labels"])
    pred = lexical_predict(record["context"], record["output"])
    scored.append((f1_from_masks(pred, gold)["f1"], record, pred))

best_f1, best_record, best_pred = max(scored, key=lambda item: item[0])
predicted = dict(best_record)
predicted["hallucination_labels"] = mask_to_spans(best_record["output"], best_pred)
predicted["meta"] = {"corruption_type": "lexical_pred"}

print(f"Example lexical F1: {best_f1:.3f}")
render_record(best_record, "Gold labels")
render_record(predicted, "Lexical baseline prediction")

Example lexical F1: 0.886


,label,start,end,len,text
0,overgeneration,0,461,461,"Indeed, both Dubai and London offer unique cul..."


,label,start,end,len,text
0,lexical_pred,0,6,6,Indeed
1,lexical_pred,8,12,4,both
2,lexical_pred,13,18,5,Dubai
3,lexical_pred,19,22,3,and
4,lexical_pred,23,29,6,London
...,...,...,...,...,...
60,lexical_pred,431,435,4,help
61,lexical_pred,436,440,4,with
62,lexical_pred,441,449,8,anything
63,lexical_pred,450,454,4,else


## 10. Reproduction Commands

```bash
python -m src.data_processing.build_from_toolace

python -m src.data_processing.audit summary   --decisions data/quality_audit_openrouter/combined/train/decisions.jsonl   --source data/combined/train.jsonl

python -m src.data_processing.recover cleans   --decisions data/quality_audit_openrouter/combined/train/decisions.jsonl   --source data/combined/train.jsonl   --out-dir data/recovered --split train
python -m src.data_processing.recover extra-spans
python -m src.data_processing.recover other

python -m src.data_processing.merge_final
python -m src.data_processing.validate_spans --allow-clean   data/final/combined/*.jsonl   data/final/contradiction/*.jsonl   data/final/missing_tool/*.jsonl   data/final/overgeneration/*.jsonl

python -m src.data_processing.zero_shot_eval   --dataset-dir data/final/combined --split validation --no-model
```